## Amar Passport Ai Agent
-Applied AI with Multi-Agent Systems (CrewAI)

In [ ]:
# Install required packages
!pip install -q crewai langchain-groq litellm

In [ ]:
# Warning control & importing libraries
import warnings
warnings.filterwarnings('ignore')
import os
import json
import logging
os.environ["CREWAI_TRACING_ENABLED"] = "false"
from crewai import Agent, Task, Crew, Process
from IPython.display import Markdown, display


## Setup APi Key

In [ ]:
import getpass
logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)

# Load Groq API key
if not os.environ.get("GROQ_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GROQ_API_KEY"] = userdata.get("Groq_Api_Key")
        print("API key loaded from Colab Secrets.")
    except Exception:
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")
        print("API key loaded via manual input.")
else:
    print("API key already set in environment.")

llm = "groq/meta-llama/llama-4-scout-17b-16e-instruct"

print("LLM ready: llama-4-scout-17b-16e-instruct via Groq (300K TPM)")

API key loaded from Colab Secrets.
LLM ready: llama-4-scout-17b-16e-instruct via Groq (300K TPM)


In [ ]:
# Test the LLM connection via LiteLLM
from litellm import completion

response = completion(
    model=llm,
    messages=[{"role": "user", "content": "Hello"}]
)
print(response.choices[0].message.content)

Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?


## Local Fallback Database


In [ ]:
# Local fallback database - 2026 official fee structure
LOCAL_PASSPORT_DB = {
    "fees_2026": {
        "48_pages": {
            "5_years":  {"regular": 4025,  "express": 6325,  "super_express": 8625},
            "10_years": {"regular": 5750,  "express": 8050,  "super_express": 10350}
        },
        "64_pages": {
            "5_years":  {"regular": 6325,  "express": 8625,  "super_express": 12075},
            "10_years": {"regular": 8050,  "express": 10350, "super_express": 13800}
        }
    },
    "required_docs": {
        "adult":            ["NID Card", "Application Summary", "Payment Slip"],
        "minor_under_18":   ["Birth Registration (English)", "Parents NID", "3R Photo"],
        "government_staff": ["NOC (No Objection Certificate)", "NID", "Application Summary", "Payment Slip"],
        "private_sector":   ["NID Card", "Profession Proof / Employment Letter", "Application Summary", "Payment Slip"],
        "name_change":      ["Marriage Certificate", "NID Card", "Application Summary", "Payment Slip"],
        "senior_above_65":  ["NID Card", "Application Summary", "Payment Slip"]
    },
    "policy_rules": {
        "under_18":     {"max_validity": "5 years",  "id_required": "Birth Registration (English)", "note": "Parents NID mandatory. 10-year passport NOT permitted."},
        "18_to_64":     {"max_validity": "10 years", "id_required": "NID Card",                    "note": "Standard eligibility. Both 5-year and 10-year are permitted."},
        "65_and_above": {"max_validity": "5 years",  "id_required": "NID Card",                    "note": "Senior citizens are capped at 5-year validity only."}
    },
    "delivery_time": {
        "regular": "15 Working Days",
        "express": "7 Working Days",
        "super_express": "2 Working Days (Pickup from Agargaon, Dhaka)"
     },
    "super_express_rule": "Available only inside Bangladesh."
}

DB_STRING = json.dumps(LOCAL_PASSPORT_DB, indent=2)
print("Local fallback database loaded.")
print(DB_STRING)

Local fallback database loaded.
{
  "fees_2026": {
    "48_pages": {
      "5_years": {
        "regular": 4025,
        "express": 6325,
        "super_express": 8625
      },
      "10_years": {
        "regular": 5750,
        "express": 8050,
        "super_express": 10350
      }
    },
    "64_pages": {
      "5_years": {
        "regular": 6325,
        "express": 8625,
        "super_express": 12075
      },
      "10_years": {
        "regular": 8050,
        "express": 10350,
        "super_express": 13800
      }
    }
  },
  "required_docs": {
    "adult": [
      "NID Card",
      "Application Summary",
      "Payment Slip"
    ],
    "minor_under_18": [
      "Birth Registration (English)",
      "Parents NID",
      "3R Photo"
    ],
    "government_staff": [
      "NOC (No Objection Certificate)",
      "NID",
      "Application Summary",
      "Payment Slip"
    ],
    "private_sector": [
      "NID Card",
      "Profession Proof / Employment Letter",
      "Applicatio

## Creating Agents



In [ ]:
# Agent 1: The Policy Guardian (Eligibility Agent)

policy_guardian = Agent(
    role="Bangladesh Passport Policy Expert",
    goal=(
        "Determine the permitted passport validity (5 years or 10 years) and the required "
        "identification document (NID or Birth Registration) based on the applicant age. "
        "Flag any inconsistency immediately, such as a minor under 18 requesting a 10-year passport "
        "or a senior above 65 requesting a 10-year passport."
    ),
    backstory=(
        "You are a senior official at the Department of Immigration and Passports, Bangladesh, "
        "with 20 years of experience enforcing passport eligibility policy. "
        "Rules: applicants under 18 are limited to 5-year passports and must present a Birth "
        "Registration in English (not NID); both parents' NID cards are also mandatory. "
        "Citizens aged 65 and above are restricted to 5-year validity only. "
        "Adults aged 18 to 64 qualify for either 5-year or 10-year passports and must present NID. "
        "When you detect an eligibility violation you write: "
        "ELIGIBILITY FLAG: followed by a clear explanation and the corrected valid option."
    ),
    allow_delegation=False,
    max_iter=5,
    llm=llm
)

In [ ]:
# Agent 2: The Chancellor of the Exchequer (Fee Calculator)

fee_calculator = Agent(
    role="Financial Auditor",
    goal=(
        "Calculate the exact BDT fee for the passport application using the 2026 official fee structure. "
        "The fee depends on page count (48 or 64 pages), delivery speed (Regular, Express, Super Express), "
        "and passport validity (5 or 10 years). All fees already include 15% VAT. "
        "If the Policy Guardian flagged an eligibility error, use the corrected validity for the fee."
    ),
    backstory=(
        "You are a certified financial auditor at the Bangladesh Passport Office with full knowledge "
        "of the 2026 official fee schedule (15% VAT already included in all prices):\n"
        "48-page / 5-year:  Regular 4025, Express 6325, Super Express 8625 BDT\n"
        "48-page / 10-year: Regular 5750, Express 8050, Super Express 10350 BDT\n"
        "64-page / 5-year:  Regular 6325, Express 8625, Super Express 12075 BDT\n"
        "64-page / 10-year: Regular 8050, Express 10350, Super Express 13800 BDT\n"
        "Delivery times: Regular=15 working days, Express=7 working days, "
        "Super Express=2 working days (Agargaon pickup only, inside Bangladesh)."
    ),
    allow_delegation=False,
    max_iter=5,
    llm=llm
)

In [ ]:
# Agent 3: The Document Architect (Checklist Specialist)
document_architect = Agent(
    role="Documentation Officer",
    goal=(
        "Generate a fully customized numbered document checklist for the passport applicant "
        "based on their age, profession, and reason for application. "
        "Include profession-specific documents: GO/NOC for government employees, "
        "Marriage Certificate for name changes, Parents' NID for minors."
    ),
    backstory=(
        "You are a documentation officer at the Bangladesh E-Passport Service Centre who has "
        "processed thousands of applications. Required documents by category:\n"
        "Minors (under 18): Birth Registration (English), both Parents' NID, 3R Photo.\n"
        "Government employees: NOC or GO (MANDATORY), NID, Application Summary, Payment Slip.\n"
        "Private sector: NID, Employment Letter or Profession Proof, Application Summary, Payment Slip.\n"
        "Name change (marriage): Marriage Certificate, NID, Application Summary, Payment Slip.\n"
        "Senior citizens (65+): NID, Application Summary, Payment Slip.\n"
        "Standard adults: NID, Application Summary, Payment Slip.\n"
        "You always produce a numbered checklist."
    ),
    allow_delegation=False,
    max_iter=5,
    llm=llm
)

In [ ]:
# Agent 4: The Report Compiler Bilingual (English + Bangla)

report_compiler = Agent(
    role="Bilingual Passport Report Officer",
    goal=(
        "Compile all outputs from the Policy Guardian, Fee Calculator, and Document Architect "
        "into a single Passport Readiness Report in BOTH English and Bangla.\n"
        "SECTION 1 — English Markdown Table: two columns Field | Details.\n"
        "SECTION 2 — Bangla Markdown Table (বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট): two columns ক্ষেত্র | বিবরণ.\n"
        "If an eligibility flag was raised, it must appear as the first row in BOTH tables."
    ),
    backstory=(
        "You are the final reporting officer at the Bangladesh Passport Authority, "
        "fluent in both English and Bangla. You compile structured information into a "
        "clean BILINGUAL Passport Readiness Report.\n\n"
        "SECTION 1 rows (English): Eligibility Flag (if any), Applicant Profile, "
        "Eligibility Status, Passport Validity, Required ID Document, Page Count, "
        "Delivery Type, Delivery Time, Total Fee (BDT VAT Included), Required Documents.\n\n"
        "SECTION 2 rows (Bangla): যোগ্যতার সমস্যা (if any), আবেদনকারীর প্রোফাইল, "
        "যোগ্যতার অবস্থা, পাসপোর্টের মেয়াদ, প্রয়োজনীয় পরিচয়পত্র, পৃষ্ঠা সংখ্যা, "
        "ডেলিভারির ধরন, ডেলিভারির সময়, মোট ফি (ভ্যাটসহ BDT), প্রয়োজনীয় কাগজপত্র.\n\n"
        "Output ONLY the two table sections with headings. No extra text outside the tables."
    ),
    allow_delegation=False,
    max_iter=5,
    llm=llm
)

###Task creation

In [ ]:
def build_tasks(user_input: str):

    # Task 1: Policy Guardian checks eligibility
    task_eligibility = Task(
        description=(
            f"Applicant profile: {user_input}\n\n"
            "Based on the applicant age, determine:\n"
            "1. Permitted passport validity: 5 years or 10 years.\n"
            "   - Under 18: maximum 5 years only. 10-year passport is NOT permitted.\n"
            "   - 18 to 64: both 5-year and 10-year are permitted.\n"
            "   - 65 and above: maximum 5 years only. 10-year passport is NOT permitted.\n"
            "2. Required identification: NID Card (adults) or Birth Registration in English (minors).\n"
            "3. Whether the requested validity is allowed for this age group.\n"
            "4. If the request is INVALID (e.g., minor/senior requesting 10-year passport), "
            "   write 'ELIGIBILITY FLAG:' followed by a clear explanation and the corrected valid option.\n"
            "Use the local database as fallback if needed."
        ),
        expected_output=(
            "A structured eligibility decision containing: "
            "permitted validity, required ID document, eligibility status (Approved or Flagged), "
            "and a flag message if applicable."
        ),
        agent=policy_guardian
    )

    # Task 2: Fee Calculator computes the fee
    # passes the Policy Guardian output as context (Task Delegation requirement)
    task_fee = Task(
        description=(
            f"Applicant profile: {user_input}\n\n"
            "Using the eligibility decision from the Policy Guardian as context:\n"
            "1. Identify the page count requested (48 or 64 pages).\n"
            "2. Identify the delivery speed (Regular, Express, or Super Express).\n"
            "3. Use the corrected validity from the eligibility decision "
            "   (if the Policy Guardian changed it, use the corrected value).\n"
            "4. Look up the exact fee from the 2026 fee structure in the local database.\n"
            "5. State clearly that the fee includes 15% VAT (fees in the 2026 structure are VAT-inclusive).\n"
            "6. Also state the expected delivery time for the chosen delivery type.\n"
            "Use the local database as fallback if needed."
        ),
        expected_output=(
            "A fee breakdown containing: page count, delivery type, validity used, "
            "delivery time, and total fee in BDT with confirmation that 15% VAT is included."
        ),
        agent=fee_calculator,
        context=[task_eligibility]
    )

    # Task 3: Document Architect builds the checklist
    task_documents = Task(
        description=(
            f"Applicant profile: {user_input}\n\n"
            "Based on the applicant age, profession, and special circumstances:\n"
            "1. Determine the applicant category (minor under 18, adult, government employee, "
            "   private sector, senior above 65, name change).\n"
            "2. Generate a numbered document checklist customised for this applicant.\n"
            "3. Include profession-specific documents:\n"
            "   - Government employees: GO or NOC (No Objection Certificate) is MANDATORY.\n"
            "   - Private sector: Employment Letter or Profession Proof.\n"
            "   - Minors: Birth Registration (English), both Parents' NID, 3R Photo.\n"
            "   - Name change due to marriage: Marriage Certificate.\n"
            "   - Senior citizens (65+): NID, Application Summary, Payment Slip.\n"
            "Use the local database as fallback if needed."
        ),
        expected_output=(
            "A numbered checklist of all required documents customised to the applicant profile."
        ),
        agent=document_architect,
        context=[task_eligibility]
    )

    # Task 4: Report Compiler produces the final BILINGUAL report
    # output must be a Markdown Table in English AND Bangla
    task_report = Task(
        description=(
            f"Applicant profile: {user_input}\n\n"
            "Using ALL outputs from the Policy Guardian, Fee Calculator, and Document Architect, "
            "compile the final Passport Readiness Report in BOTH English and Bangla.\n\n"
            "--- SECTION 1: English Markdown Table ---\n"
            "Two columns: Field | Details.\n"
            "Rows: Applicant Profile, Eligibility Status, Passport Validity, "
            "Required ID Document, Page Count, Delivery Type, Delivery Time, "
            "Total Fee (BDT VAT Included), Required Documents.\n"
            "If an ELIGIBILITY FLAG was raised, add it as the FIRST row.\n\n"
            "--- SECTION 2: বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট ---\n"
            "Translate the entire table into Bangla. Two columns: ক্ষেত্র | বিবরণ.\n"
            "All row labels AND all content values must be written in Bangla.\n"
            "If an eligibility flag exists, first row label must be: যোগ্যতার সমস্যা\n\n"
            "Output ONLY the two table sections with headings. No extra text outside the tables."
        ),
        expected_output=(
            "A complete bilingual Passport Readiness Report: "
            "Section 1 as an English Markdown Table, "
            "Section 2 as a Bangla Markdown Table (বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট)."
        ),
        agent=report_compiler,
        context=[task_eligibility, task_fee, task_documents]
    )

    return [task_eligibility, task_fee, task_documents, task_report]

In [ ]:
#creating crew
def build_crew(user_input: str) -> Crew:
    tasks = build_tasks(user_input)
    crew = Crew(
        agents=[policy_guardian, fee_calculator, document_architect, report_compiler],
        tasks=tasks,
        verbose=True
    )
    return crew

In [ ]:
# Standard Adult Applicant
scenario_1 = (
    "I am a 24-year-old private sector employee. "
    "I need a 64-page passport with Express delivery because I have a business trip in two weeks. "
    "I have an NID and I live in Dhaka."
)

crew_1 = build_crew(scenario_1)
result_1 = crew_1.kickoff()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  d4cf5ac3-b119-4e4c-8171-fdc64d348ab7                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Applicant profile: I am a 24-year-old private sector employee. I need a 64-page passport with Express    │
│  delivery because I have a business trip in two weeks. I have an NID and I live in Dhaka.                       │
│                                                                                                                 │
│  Based on the applicant age, determine:                                                                         │
│  1. Permitted passport validity: 5 years or 10 years.                                                           │
│     - Under 18: maximum 5 years only. 10-year passport is NOT permitted.                                        │
│     - 18 to 64: both 5-year and 10-year are permitted.                                                          │
│     - 65 and above: maximum 5 years only. 10-year passport is NOT permitted.                                    │
│  2. Required identification: NID Card (adults) or Birth Registration in English (minors).                       │
│  3. Whether the requested validity is allowed for this age group.                                               │
│  4. If the request is INVALID (e.g., minor/senior requesting 10-year passport),    write 'ELIGIBILITY FLAG:'    │
│  followed by a clear explanation and the corrected valid option.                                                │
│  Use the local database as fallback if needed.                                                                  │
│  ID: e9e8e86b-ece1-4aa4-b861-2e16e1171e5e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bangladesh Passport Policy Expert                                                                       │
│                                                                                                                 │
│  Task: Applicant profile: I am a 24-year-old private sector employee. I need a 64-page passport with Express    │
│  delivery because I have a business trip in two weeks. I have an NID and I live in Dhaka.                       │
│                                                                                                                 │
│  Based on the applicant age, determine:                                                                         │
│  1. Permitted passport validity: 5 years or 10 years.                                                           │
│     - Under 18: maximum 5 years only. 10-year passport is NOT permitted.                                        │
│     - 18 to 64: both 5-year and 10-year are permitted.                                                          │
│     - 65 and above: maximum 5 years only. 10-year passport is NOT permitted.                                    │
│  2. Required identification: NID Card (adults) or Birth Registration in English (minors).                       │
│  3. Whether the requested validity is allowed for this age group.                                               │
│  4. If the request is INVALID (e.g., minor/senior requesting 10-year passport),    write 'ELIGIBILITY FLAG:'    │
│  followed by a clear explanation and the corrected valid option.                                                │
│  Use the local database as fallback if needed.                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bangladesh Passport Policy Expert                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Based on the applicant's profile:                                                                              │
│                                                                                                                 │
│  1. **Permitted passport validity**: Since the applicant is 24 years old, which falls within the 18 to 64 age   │
│  range, they are eligible for either a 5-year or 10-year passport validity.                                     │
│                                                                                                                 │
│  2. **Required identification**: As the applicant is an adult (above 18 years old), the required                │
│  identification document is an NID Card.                                                                        │
│                                                                                                                 │
│  3. **Requested validity and allowance**: The applicant did not specify the requested validity but mentioned a  │
│  need for a passport urgently. Given their age, they can opt for either a 5-year or 10-year passport.           │
│                                                                                                                 │
│  4. **Eligibility status**: The applicant's request seems to be generally in line with the eligibility          │
│  criteria based on age. However, the specific request for a "64-page passport" isn't directly addressed in the  │
│  standard criteria but can be considered under normal procedures for a standard passport application. The       │
│  urgency (Express delivery) also doesn't affect eligibility directly.                                           │
│                                                                                                                 │
│  Given the provided information and standard criteria:                                                          │
│                                                                                                                 │
│  - **Permitted validity**: 5 years or 10 years                                                                  │
│  - **Required ID document**: NID Card                                                                           │
│  - **Eligibility status**: Approved                                                                             │
│  - **Flag message**: None                                                                                       │
│                                                                                                                 │
│  **Final Eligibility Decision:**                                                                                │
│  - Permitted validity: 5 years or 10 years                                                                      │
│  - Required ID document: NID Card                                                                               │
│  - Eligibility status: Approved                                                                                 │
│  - Flag message: None                                                                                           │
│                                                                                                                 │
│  The applicant, being 24 years old and having an NID, i

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Applicant profile: I am a 24-year-old private sector employee. I need a 64-page passport with Express          │
│  delivery because I have a business trip in two weeks. I have an NID and I live in Dhaka.                       │
│                                                                                                                 │
│  Based on the applicant age, determine:                                                                         │
│  1. Permitted passport validity: 5 years or 10 years.                                                           │
│     - Under 18: maximum 5 years only. 10-year passport is NOT permitted.                                        │
│     - 18 to 64: both 5-year and 10-year are permitted.                                                          │
│     - 65 and above: maximum 5 years only. 10-year passport is NOT permitted.                                    │
│  2. Required identification: NID Card (adults) or Birth Registration in English (minors).                       │
│  3. Whether the requested validity is allowed for this age group.                                               │
│  4. If the request is INVALID (e.g., minor/senior requesting 10-year passport),    write 'ELIGIBILITY FLAG:'    │
│  followed by a clear explanation and the corrected valid option.                                                │
│  Use the local database as fallback if needed.                                                                  │
│  Agent:                                                                                                         │
│  Bangladesh Passport Policy Expert                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Applicant profile: I am a 24-year-old private sector employee. I need a 64-page passport with Express    │
│  delivery because I have a business trip in two weeks. I have an NID and I live in Dhaka.                       │
│                                                                                                                 │
│  Using ALL outputs from the Policy Guardian, Fee Calculator, and Document Architect, compile the final          │
│  Passport Readiness Report in BOTH English and Bangla.                                                          │
│                                                                                                                 │
│  --- SECTION 1: English Markdown Table ---                                                                      │
│  Two columns: Field | Details.                                                                                  │
│  Rows: Applicant Profile, Eligibility Status, Passport Validity, Required ID Document, Page Count, Delivery     │
│  Type, Delivery Time, Total Fee (BDT VAT Included), Required Documents.                                         │
│  If an ELIGIBILITY FLAG was raised, add it as the FIRST row.                                                    │
│                                                                                                                 │
│  --- SECTION 2: বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট ---                                                                         │
│  Translate the entire table into Bangla. Two columns: ক্ষেত্র | বিবরণ.                                              │
│  All row labels AND all content values must be written in Bangla.                                               │
│  If an eligibility flag exists, first row label must be: যোগ্যতার সমস্যা                                             │
│                                                                                                                 │
│  Output ONLY the two table sections with headings. No extra text outside the tables.                            │
│  ID: 7706ddca-f083-4983-9ec9-9e1082fa59fb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bilingual Passport Report Officer                                                                       │
│                                                                                                                 │
│  Task: Applicant profile: I am a 24-year-old private sector employee. I need a 64-page passport with Express    │
│  delivery because I have a business trip in two weeks. I have an NID and I live in Dhaka.                       │
│                                                                                                                 │
│  Using ALL outputs from the Policy Guardian, Fee Calculator, and Document Architect, compile the final          │
│  Passport Readiness Report in BOTH English and Bangla.                                                          │
│                                                                                                                 │
│  --- SECTION 1: English Markdown Table ---                                                                      │
│  Two columns: Field | Details.                                                                                  │
│  Rows: Applicant Profile, Eligibility Status, Passport Validity, Required ID Document, Page Count, Delivery     │
│  Type, Delivery Time, Total Fee (BDT VAT Included), Required Documents.                                         │
│  If an ELIGIBILITY FLAG was raised, add it as the FIRST row.                                                    │
│                                                                                                                 │
│  --- SECTION 2: বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট ---                                                                         │
│  Translate the entire table into Bangla. Two columns: ক্ষেত্র | বিবরণ.                                              │
│  All row labels AND all content values must be written in Bangla.                                               │
│  If an eligibility flag exists, first row label must be: যোগ্যতার সমস্যা                                             │
│                                                                                                                 │
│  Output ONLY the two table sections with headings. No extra text outside the tables.                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bilingual Passport Report Officer                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### SECTION 1 — English Markdown Table                                                                         │
│  | Field | Details |                                                                                            │
│  | --- | --- |                                                                                                  │
│  | Eligibility Flag |  |                                                                                        │
│  | Applicant Profile | 24-year-old private sector employee |                                                    │
│  | Eligibility Status | Approved |                                                                              │
│  | Passport Validity | 5 years or 10 years |                                                                    │
│  | Required ID Document | NID Card |                                                                            │
│  | Page Count | 64 pages |                                                                                      │
│  | Delivery Type | Express |                                                                                    │
│  | Delivery Time | 7 working days |                                                                             │
│  | Total Fee (BDT VAT Included) | 8625 BDT |                                                                    │
│  | Required Documents | NID, Employment Letter or Profession Proof, Application Summary, Payment Slip |         │
│                                                                                                                 │
│  ### SECTION 2 — বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট                                                                            │
│  | ক্ষেত্র | বিবরণ |                                                                                                │
│  | --- | --- |                                                                                                  │
│  | যোগ্যতার সমস্যা |  |                                                                                              │
│  | আবেদনকারীর প্রোফাইল | ২৪ বছর বয়সী বেসরকারী খাতের কর্মচারী |                                                                 │
│  | যোগ্যতার অবস্থা | অনুমোদিত |                                                                                         │
│  | পাসপোর্টের মেয়াদ | ৫ বছর বা ১০ বছর |                                                                                │
│  | প্রয়োজনীয় পরিচয়পত্র | এনআইডি কার্ড |                                                                                 │
│  | পৃষ্ঠা সংখ্যা | ৬৪ পৃষ্ঠা |                                                                                           │
│  | ডেলিভারির ধরন | এক্সপ্রেস |                                                                                         │
│  | ডেলিভারির সময় | ৭ কর্মদিবস |                                                                                       │
│  | মোট ফি (ভ্যাটসহ BDT) | ৮৬২৫ BDT |                                                                                │
│  | প্রয়োজনীয় কাগজপত্র | এনআইডি, চাকুরির প্রমাণপত্র বা পেশা প্রমাণ, আবেদন সারাংশ, পেমেন্ট স্লিপ |                                          │
│                                                                                                                 │
╰────────────────────────────────────────────────────

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Applicant profile: I am a 24-year-old private sector employee. I need a 64-page passport with Express          │
│  delivery because I have a business trip in two weeks. I have an NID and I live in Dhaka.                       │
│                                                                                                                 │
│  Using ALL outputs from the Policy Guardian, Fee Calculator, and Document Architect, compile the final          │
│  Passport Readiness Report in BOTH English and Bangla.                                                          │
│                                                                                                                 │
│  --- SECTION 1: English Markdown Table ---                                                                      │
│  Two columns: Field | Details.                                                                                  │
│  Rows: Applicant Profile, Eligibility Status, Passport Validity, Required ID Document, Page Count, Delivery     │
│  Type, Delivery Time, Total Fee (BDT VAT Included), Required Documents.                                         │
│  If an ELIGIBILITY FLAG was raised, add it as the FIRST row.                                                    │
│                                                                                                                 │
│  --- SECTION 2: বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট ---                                                                         │
│  Translate the entire table into Bangla. Two columns: ক্ষেত্র | বিবরণ.                                              │
│  All row labels AND all content values must be written in Bangla.                                               │
│  If an eligibility flag exists, first row label must be: যোগ্যতার সমস্যা                                             │
│                                                                                                                 │
│  Output ONLY the two table sections with headings. No extra text outside the tables.                            │
│  Agent:                                                                                                         │
│  Bilingual Passport Report Officer                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  d4cf5ac3-b119-4e4c-8171-fdc64d348ab7                                                                           │
│  Final Output: ### SECTION 1 — English Markdown Table                                                           │
│  | Field | Details |                                                                                            │
│  | --- | --- |                                                                                                  │
│  | Eligibility Flag |  |                                                                                        │
│  | Applicant Profile | 24-year-old private sector employee |                                                    │
│  | Eligibility Status | Approved |                                                                              │
│  | Passport Validity | 5 years or 10 years |                                                                    │
│  | Required ID Document | NID Card |                                                                            │
│  | Page Count | 64 pages |                                                                                      │
│  | Delivery Type | Express |                                                                                    │
│  | Delivery Time | 7 working days |                                                                             │
│  | Total Fee (BDT VAT Included) | 8625 BDT |                                                                    │
│  | Required Documents | NID, Employment Letter or Profession Proof, Application Summary, Payment Slip |         │
│                                                                                                                 │
│  ### SECTION 2 — বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট                                                                            │
│  | ক্ষেত্র | বিবরণ |                                                                                                │
│  | --- | --- |                                                                                                  │
│  | যোগ্যতার সমস্যা |  |                                                                                              │
│  | আবেদনকারীর প্রোফাইল | ২৪ বছর বয়সী বেসরকারী খাতের কর্মচারী |                                                                 │
│  | যোগ্যতার অবস্থা | অনুমোদিত |                                                                                         │
│  | পাসপোর্টের মেয়াদ | ৫ বছর বা ১০ বছর |                                                                                │
│  | প্রয়োজনীয় পরিচয়পত্র | এনআইডি কার্ড |                                                                                 │
│  | পৃষ্ঠা সংখ্যা | ৬৪ পৃষ্ঠা |                                                                                           │
│  | ডেলিভারির ধরন | এক্সপ্রেস |                                                                                         │
│  | ডেলিভারির সময় | ৭ কর্মদিবস |                                                                                       │
│  | মোট ফি (ভ্যাটসহ BDT) | ৮৬২৫ BDT |                                                                                │
│  | প্রয়োজনীয় কাগজপত্র | এনআইডি, চাকুরির প্রমাণপত্র বা পেশা প্রমাণ, আবেদন সার

In [ ]:
# Display the bilingual report
display(Markdown(str(result_1)))


### SECTION 1 — English Markdown Table 
| Field | Details |
| --- | --- |
| Eligibility Flag |  |
| Applicant Profile | 24-year-old private sector employee |
| Eligibility Status | Approved |
| Passport Validity | 5 years or 10 years |
| Required ID Document | NID Card |
| Page Count | 64 pages |
| Delivery Type | Express |
| Delivery Time | 7 working days |
| Total Fee (BDT VAT Included) | 8625 BDT |
| Required Documents | NID, Employment Letter or Profession Proof, Application Summary, Payment Slip |

### SECTION 2 — বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট 
| ক্ষেত্র | বিবরণ |
| --- | --- |
| যোগ্যতার সমস্যা |  |
| আবেদনকারীর প্রোফাইল | ২৪ বছর বয়সী বেসরকারী খাতের কর্মচারী |
| যোগ্যতার অবস্থা | অনুমোদিত |
| পাসপোর্টের মেয়াদ | ৫ বছর বা ১০ বছর |
| প্রয়োজনীয় পরিচয়পত্র | এনআইডি কার্ড |
| পৃষ্ঠা সংখ্যা | ৬৪ পৃষ্ঠা |
| ডেলিভারির ধরন | এক্সপ্রেস |
| ডেলিভারির সময় | ৭ কর্মদিবস |
| মোট ফি (ভ্যাটসহ BDT) | ৮৬২৫ BDT |
| প্রয়োজনীয় কাগজপত্র | এনআইডি, চাকুরির প্রমাণপত্র বা পেশা প্রমাণ, আবেদন সারাংশ, পেমেন্ট স্লিপ |

In [ ]:
#Error Handling (Minor Requesting 10-Year Passport)
scenario_2 = (
    "I am a 15-year-old student. "
    "I want a 64-page passport with 10-year validity for traveling abroad with my family. "
    "I have a birth registration certificate. My parents have NID cards."
)

crew_2 = build_crew(scenario_2)
result_2 = crew_2.kickoff()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  1ba4c5fc-314b-41e7-b7e7-59e94aa6653d                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Applicant profile: I am a 15-year-old student. I want a 64-page passport with 10-year validity for       │
│  traveling abroad with my family. I have a birth registration certificate. My parents have NID cards.           │
│                                                                                                                 │
│  Based on the applicant age, determine:                                                                         │
│  1. Permitted passport validity: 5 years or 10 years.                                                           │
│     - Under 18: maximum 5 years only. 10-year passport is NOT permitted.                                        │
│     - 18 to 64: both 5-year and 10-year are permitted.                                                          │
│     - 65 and above: maximum 5 years only. 10-year passport is NOT permitted.                                    │
│  2. Required identification: NID Card (adults) or Birth Registration in English (minors).                       │
│  3. Whether the requested validity is allowed for this age group.                                               │
│  4. If the request is INVALID (e.g., minor/senior requesting 10-year passport),    write 'ELIGIBILITY FLAG:'    │
│  followed by a clear explanation and the corrected valid option.                                                │
│  Use the local database as fallback if needed.                                                                  │
│  ID: fbbf4eac-155b-4646-8e9d-2e48c72cac3b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bangladesh Passport Policy Expert                                                                       │
│                                                                                                                 │
│  Task: Applicant profile: I am a 15-year-old student. I want a 64-page passport with 10-year validity for       │
│  traveling abroad with my family. I have a birth registration certificate. My parents have NID cards.           │
│                                                                                                                 │
│  Based on the applicant age, determine:                                                                         │
│  1. Permitted passport validity: 5 years or 10 years.                                                           │
│     - Under 18: maximum 5 years only. 10-year passport is NOT permitted.                                        │
│     - 18 to 64: both 5-year and 10-year are permitted.                                                          │
│     - 65 and above: maximum 5 years only. 10-year passport is NOT permitted.                                    │
│  2. Required identification: NID Card (adults) or Birth Registration in English (minors).                       │
│  3. Whether the requested validity is allowed for this age group.                                               │
│  4. If the request is INVALID (e.g., minor/senior requesting 10-year passport),    write 'ELIGIBILITY FLAG:'    │
│  followed by a clear explanation and the corrected valid option.                                                │
│  Use the local database as fallback if needed.                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bangladesh Passport Policy Expert                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Based on the applicant's profile:                                                                              │
│                                                                                                                 │
│  1. **Permitted passport validity**: Since the applicant is 15 years old, which is under 18, they are limited   │
│  to a maximum of 5-year passport validity. A 10-year passport is not permitted for minors.                      │
│                                                                                                                 │
│  2. **Required identification**: As the applicant is a minor (under 18 years old), the required identification  │
│  document is a Birth Registration in English. Additionally, both parents' NID cards are mandatory.              │
│                                                                                                                 │
│  3. **Requested validity and allowance**: The applicant requested a 10-year passport validity, which is not     │
│  allowed for their age group.                                                                                   │
│                                                                                                                 │
│  4. **Eligibility status**: The request for a 10-year passport by a minor is not in line with the eligibility   │
│  criteria.                                                                                                      │
│                                                                                                                 │
│  **ELIGIBILITY FLAG:** The applicant, being 15 years old, is not eligible for a 10-year passport. According to  │
│  the regulations, minors under 18 are limited to a 5-year passport validity.                                    │
│                                                                                                                 │
│  **Corrected valid option:** The applicant can apply for a 5-year validity passport. The required documents     │
│  include a Birth Registration certificate in English and both parents' NID cards.                               │
│                                                                                                                 │
│  **Final Eligibility Decision:**                                                                                │
│  - Permitted validity: 5 years                                                                                  │
│  - Required ID document: Birth Registration in English and both parents' NID cards                              │
│  - Eligibility status: Flagged                                                                                  │
│  - Flag message: ELIGIBILITY FLAG: Minors under 18 are not eligible for a 10-year passport. The maximum         │
│  validity permitted is 5 years. Please correct the application to request a 5-year passport validity.           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Applicant profile: I am a 15-year-old student. I want a 64-page passport with 10-year validity for traveling   │
│  abroad with my family. I have a birth registration certificate. My parents have NID cards.                     │
│                                                                                                                 │
│  Based on the applicant age, determine:                                                                         │
│  1. Permitted passport validity: 5 years or 10 years.                                                           │
│     - Under 18: maximum 5 years only. 10-year passport is NOT permitted.                                        │
│     - 18 to 64: both 5-year and 10-year are permitted.                                                          │
│     - 65 and above: maximum 5 years only. 10-year passport is NOT permitted.                                    │
│  2. Required identification: NID Card (adults) or Birth Registration in English (minors).                       │
│  3. Whether the requested validity is allowed for this age group.                                               │
│  4. If the request is INVALID (e.g., minor/senior requesting 10-year passport),    write 'ELIGIBILITY FLAG:'    │
│  followed by a clear explanation and the corrected valid option.                                                │
│  Use the local database as fallback if needed.                                                                  │
│  Agent:                                                                                                         │
│  Bangladesh Passport Policy Expert                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Applicant profile: I am a 15-year-old student. I want a 64-page passport with 10-year validity for       │
│  traveling abroad with my family. I have a birth registration certificate. My parents have NID cards.           │
│                                                                                                                 │
│  Using ALL outputs from the Policy Guardian, Fee Calculator, and Document Architect, compile the final          │
│  Passport Readiness Report in BOTH English and Bangla.                                                          │
│                                                                                                                 │
│  --- SECTION 1: English Markdown Table ---                                                                      │
│  Two columns: Field | Details.                                                                                  │
│  Rows: Applicant Profile, Eligibility Status, Passport Validity, Required ID Document, Page Count, Delivery     │
│  Type, Delivery Time, Total Fee (BDT VAT Included), Required Documents.                                         │
│  If an ELIGIBILITY FLAG was raised, add it as the FIRST row.                                                    │
│                                                                                                                 │
│  --- SECTION 2: বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট ---                                                                         │
│  Translate the entire table into Bangla. Two columns: ক্ষেত্র | বিবরণ.                                              │
│  All row labels AND all content values must be written in Bangla.                                               │
│  If an eligibility flag exists, first row label must be: যোগ্যতার সমস্যা                                             │
│                                                                                                                 │
│  Output ONLY the two table sections with headings. No extra text outside the tables.                            │
│  ID: 53302f26-ad1a-4d64-8d4d-59a94fc88607                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bilingual Passport Report Officer                                                                       │
│                                                                                                                 │
│  Task: Applicant profile: I am a 15-year-old student. I want a 64-page passport with 10-year validity for       │
│  traveling abroad with my family. I have a birth registration certificate. My parents have NID cards.           │
│                                                                                                                 │
│  Using ALL outputs from the Policy Guardian, Fee Calculator, and Document Architect, compile the final          │
│  Passport Readiness Report in BOTH English and Bangla.                                                          │
│                                                                                                                 │
│  --- SECTION 1: English Markdown Table ---                                                                      │
│  Two columns: Field | Details.                                                                                  │
│  Rows: Applicant Profile, Eligibility Status, Passport Validity, Required ID Document, Page Count, Delivery     │
│  Type, Delivery Time, Total Fee (BDT VAT Included), Required Documents.                                         │
│  If an ELIGIBILITY FLAG was raised, add it as the FIRST row.                                                    │
│                                                                                                                 │
│  --- SECTION 2: বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট ---                                                                         │
│  Translate the entire table into Bangla. Two columns: ক্ষেত্র | বিবরণ.                                              │
│  All row labels AND all content values must be written in Bangla.                                               │
│  If an eligibility flag exists, first row label must be: যোগ্যতার সমস্যা                                             │
│                                                                                                                 │
│  Output ONLY the two table sections with headings. No extra text outside the tables.                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bilingual Passport Report Officer                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### SECTION 1 — English Markdown Table                                                                         │
│  | Field | Details |                                                                                            │
│  | --- | --- |                                                                                                  │
│  | Eligibility Flag | ELIGIBILITY FLAG: Minors under 18 are not eligible for a 10-year passport. The maximum    │
│  validity permitted is 5 years. |                                                                               │
│  | Applicant Profile | 15-year-old student |                                                                    │
│  | Eligibility Status | Flagged |                                                                               │
│  | Passport Validity | 5 years (Corrected from 10 years) |                                                      │
│  | Required ID Document | Birth Registration in English and both parents' NID cards |                           │
│  | Page Count | 64 pages |                                                                                      │
│  | Delivery Type | Regular |                                                                                    │
│  | Delivery Time | 15 working days |                                                                            │
│  | Total Fee (BDT VAT Included) | 6325 BDT |                                                                    │
│  | Required Documents | Birth Registration (English), Both Parents' NID Cards, 3R Photo, Application Summary,   │
│  Payment Slip |                                                                                                 │
│                                                                                                                 │
│  ### SECTION 2 — বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট                                                                            │
│  | ক্ষেত্র | বিবরণ |                                                                                                │
│  | --- | --- |                                                                                                  │
│  | যোগ্যতার সমস্যা | যোগ্যতার সমস্যা: ১৮ বছরের নিচের আবেদনকারীরা ১০ বছরের পাসপোর্টের জন্য যোগ্য নয়। সর্বোচ্চ অনুমোদিত মেয়াদ ৫ বছর। |             │
│  | আবেদনকারীর প্রোফাইল | ১৫ বছর বয়সী ছাত্র |                                                                             │
│  | যোগ্যতার অবস্থা | ফ্ল্যাগযুক্ত |                                                                                       │
│  | পাসপোর্টের মেয়াদ | ৫ বছর (১০ বছর থেকে সংশোধিত) |                                                                        │
│  | প্রয়োজনীয় পরিচয়পত্র | জন্ম নিবন্ধন (ইংরেজি) এবং উভয় পিতামাতার এনআইডি কার্ড |                                                    │
│  | পৃষ্ঠা সংখ্যা | ৬৪ পৃষ্ঠা |                                                                                           │
│  | ডেলিভারির ধরন | নিয়মিত |                                                                                           │
│  | ডেলিভারির সময় | ১৫ কর্মদিবস |                                                                                      │
│  | মোট ফি (ভ্যাটসহ BDT) | ৬৩২৫ BDT |                                                                                │
│  | প্রয়োজনীয় কাগজপত্র | জন্ম নিবন্ধ

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Applicant profile: I am a 15-year-old student. I want a 64-page passport with 10-year validity for traveling   │
│  abroad with my family. I have a birth registration certificate. My parents have NID cards.                     │
│                                                                                                                 │
│  Using ALL outputs from the Policy Guardian, Fee Calculator, and Document Architect, compile the final          │
│  Passport Readiness Report in BOTH English and Bangla.                                                          │
│                                                                                                                 │
│  --- SECTION 1: English Markdown Table ---                                                                      │
│  Two columns: Field | Details.                                                                                  │
│  Rows: Applicant Profile, Eligibility Status, Passport Validity, Required ID Document, Page Count, Delivery     │
│  Type, Delivery Time, Total Fee (BDT VAT Included), Required Documents.                                         │
│  If an ELIGIBILITY FLAG was raised, add it as the FIRST row.                                                    │
│                                                                                                                 │
│  --- SECTION 2: বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট ---                                                                         │
│  Translate the entire table into Bangla. Two columns: ক্ষেত্র | বিবরণ.                                              │
│  All row labels AND all content values must be written in Bangla.                                               │
│  If an eligibility flag exists, first row label must be: যোগ্যতার সমস্যা                                             │
│                                                                                                                 │
│  Output ONLY the two table sections with headings. No extra text outside the tables.                            │
│  Agent:                                                                                                         │
│  Bilingual Passport Report Officer                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  1ba4c5fc-314b-41e7-b7e7-59e94aa6653d                                                                           │
│  Final Output: ### SECTION 1 — English Markdown Table                                                           │
│  | Field | Details |                                                                                            │
│  | --- | --- |                                                                                                  │
│  | Eligibility Flag | ELIGIBILITY FLAG: Minors under 18 are not eligible for a 10-year passport. The maximum    │
│  validity permitted is 5 years. |                                                                               │
│  | Applicant Profile | 15-year-old student |                                                                    │
│  | Eligibility Status | Flagged |                                                                               │
│  | Passport Validity | 5 years (Corrected from 10 years) |                                                      │
│  | Required ID Document | Birth Registration in English and both parents' NID cards |                           │
│  | Page Count | 64 pages |                                                                                      │
│  | Delivery Type | Regular |                                                                                    │
│  | Delivery Time | 15 working days |                                                                            │
│  | Total Fee (BDT VAT Included) | 6325 BDT |                                                                    │
│  | Required Documents | Birth Registration (English), Both Parents' NID Cards, 3R Photo, Application Summary,   │
│  Payment Slip |                                                                                                 │
│                                                                                                                 │
│  ### SECTION 2 — বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট                                                                            │
│  | ক্ষেত্র | বিবরণ |                                                                                                │
│  | --- | --- |                                                                                                  │
│  | যোগ্যতার সমস্যা | যোগ্যতার সমস্যা: ১৮ বছরের নিচের আবেদনকারীরা ১০ বছরের পাসপোর্টের জন্য যোগ্য নয়। সর্বোচ্চ অনুমোদিত মেয়াদ ৫ বছর। |             │
│  | আবেদনকারীর প্রোফাইল | ১৫ বছর বয়সী ছাত্র |                                                                             │
│  | যোগ্যতার অবস্থা | ফ্ল্যাগযুক্ত |                                                                                       │
│  | পাসপোর্টের মেয়াদ | ৫ বছর (১০ বছর থেকে সংশোধিত) |                                                                        │
│  | প্রয়োজনীয় পরিচয়পত্র | জন্ম নিবন্ধন (ইংরেজি) এবং উভয় পিতামাতার এনআইডি কার্ড |                                                    │
│  | পৃষ্ঠা সংখ্যা | ৬৪ পৃষ্ঠা |                                                                                           │
│  | ডেলিভারির ধরন | নিয়মিত |                                                                                           │
│  | ডেলিভারির সময় | ১৫ কর্মদিবস |              

In [ ]:
display(Markdown(str(result_2)))


### SECTION 1 — English Markdown Table 
| Field | Details |
| --- | --- |
| Eligibility Flag | ELIGIBILITY FLAG: Minors under 18 are not eligible for a 10-year passport. The maximum validity permitted is 5 years. |
| Applicant Profile | 15-year-old student |
| Eligibility Status | Flagged |
| Passport Validity | 5 years (Corrected from 10 years) |
| Required ID Document | Birth Registration in English and both parents' NID cards |
| Page Count | 64 pages |
| Delivery Type | Regular |
| Delivery Time | 15 working days |
| Total Fee (BDT VAT Included) | 6325 BDT |
| Required Documents | Birth Registration (English), Both Parents' NID Cards, 3R Photo, Application Summary, Payment Slip |

### SECTION 2 — বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট 
| ক্ষেত্র | বিবরণ |
| --- | --- |
| যোগ্যতার সমস্যা | যোগ্যতার সমস্যা: ১৮ বছরের নিচের আবেদনকারীরা ১০ বছরের পাসপোর্টের জন্য যোগ্য নয়। সর্বোচ্চ অনুমোদিত মেয়াদ ৫ বছর। |
| আবেদনকারীর প্রোফাইল | ১৫ বছর বয়সী ছাত্র |
| যোগ্যতার অবস্থা | ফ্ল্যাগযুক্ত |
| পাসপোর্টের মেয়াদ | ৫ বছর (১০ বছর থেকে সংশোধিত) |
| প্রয়োজনীয় পরিচয়পত্র | জন্ম নিবন্ধন (ইংরেজি) এবং উভয় পিতামাতার এনআইডি কার্ড |
| পৃষ্ঠা সংখ্যা | ৬৪ পৃষ্ঠা |
| ডেলিভারির ধরন | নিয়মিত |
| ডেলিভারির সময় | ১৫ কর্মদিবস |
| মোট ফি (ভ্যাটসহ BDT) | ৬৩২৫ BDT |
| প্রয়োজনীয় কাগজপত্র | জন্ম নিবন্ধন (ইংরেজি), উভয় পিতামাতার এনআইডি কার্ড, ৩আর ফটো, আবেদন সারাংশ, পেমেন্ট স্লিপ |

In [ ]:
#Government Employee
scenario_3 = (
    "I am a 35-year-old government employee working at a public university. "
    "I need a 48-page passport with regular delivery. "
    "I have my NID card and I live in Dhaka."
)

crew_3 = build_crew(scenario_3)
result_3 = crew_3.kickoff()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f894040f-314d-42da-a7fa-6b550f28bd81                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Applicant profile: I am a 35-year-old government employee working at a public university. I need a       │
│  48-page passport with regular delivery. I have my NID card and I live in Dhaka.                                │
│                                                                                                                 │
│  Based on the applicant age, determine:                                                                         │
│  1. Permitted passport validity: 5 years or 10 years.                                                           │
│     - Under 18: maximum 5 years only. 10-year passport is NOT permitted.                                        │
│     - 18 to 64: both 5-year and 10-year are permitted.                                                          │
│     - 65 and above: maximum 5 years only. 10-year passport is NOT permitted.                                    │
│  2. Required identification: NID Card (adults) or Birth Registration in English (minors).                       │
│  3. Whether the requested validity is allowed for this age group.                                               │
│  4. If the request is INVALID (e.g., minor/senior requesting 10-year passport),    write 'ELIGIBILITY FLAG:'    │
│  followed by a clear explanation and the corrected valid option.                                                │
│  Use the local database as fallback if needed.                                                                  │
│  ID: 422bcd6b-71a0-4a35-8cba-27122e9905d4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bangladesh Passport Policy Expert                                                                       │
│                                                                                                                 │
│  Task: Applicant profile: I am a 35-year-old government employee working at a public university. I need a       │
│  48-page passport with regular delivery. I have my NID card and I live in Dhaka.                                │
│                                                                                                                 │
│  Based on the applicant age, determine:                                                                         │
│  1. Permitted passport validity: 5 years or 10 years.                                                           │
│     - Under 18: maximum 5 years only. 10-year passport is NOT permitted.                                        │
│     - 18 to 64: both 5-year and 10-year are permitted.                                                          │
│     - 65 and above: maximum 5 years only. 10-year passport is NOT permitted.                                    │
│  2. Required identification: NID Card (adults) or Birth Registration in English (minors).                       │
│  3. Whether the requested validity is allowed for this age group.                                               │
│  4. If the request is INVALID (e.g., minor/senior requesting 10-year passport),    write 'ELIGIBILITY FLAG:'    │
│  followed by a clear explanation and the corrected valid option.                                                │
│  Use the local database as fallback if needed.                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bangladesh Passport Policy Expert                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Based on the applicant's profile:                                                                              │
│                                                                                                                 │
│  1. **Permitted passport validity**: Since the applicant is 35 years old, which falls within the 18 to 64 age   │
│  range, they are eligible for either a 5-year or 10-year passport validity.                                     │
│                                                                                                                 │
│  2. **Required identification**: As the applicant is an adult (above 18 years old), the required                │
│  identification document is an NID Card.                                                                        │
│                                                                                                                 │
│  3. **Requested validity and allowance**: The applicant did not specify the requested validity but is eligible  │
│  for both. The request for a 48-page passport with regular delivery seems standard and does not inherently      │
│  conflict with eligibility criteria.                                                                            │
│                                                                                                                 │
│  4. **Eligibility status**: The applicant's request seems to be generally in line with the eligibility          │
│  criteria based on age.                                                                                         │
│                                                                                                                 │
│  Given the provided information and standard criteria:                                                          │
│                                                                                                                 │
│  - **Permitted validity**: 5 years or 10 years                                                                  │
│  - **Required ID document**: NID Card                                                                           │
│  - **Eligibility status**: Approved                                                                             │
│  - **Flag message**: None                                                                                       │
│                                                                                                                 │
│  **Final Eligibility Decision:**                                                                                │
│  - Permitted validity: 5 years or 10 years                                                                      │
│  - Required ID document: NID Card                                                                               │
│  - Eligibility status: Approved                                                                                 │
│  - Flag message: None                                                                                           │
│                                                                                                                 │
│  The applicant, being 35 years old and having an NID, is eligible for a passport with either 5-year or 10-year  │
│  validity. Their request appears to be standard and wit

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Applicant profile: I am a 35-year-old government employee working at a public university. I need a 48-page     │
│  passport with regular delivery. I have my NID card and I live in Dhaka.                                        │
│                                                                                                                 │
│  Based on the applicant age, determine:                                                                         │
│  1. Permitted passport validity: 5 years or 10 years.                                                           │
│     - Under 18: maximum 5 years only. 10-year passport is NOT permitted.                                        │
│     - 18 to 64: both 5-year and 10-year are permitted.                                                          │
│     - 65 and above: maximum 5 years only. 10-year passport is NOT permitted.                                    │
│  2. Required identification: NID Card (adults) or Birth Registration in English (minors).                       │
│  3. Whether the requested validity is allowed for this age group.                                               │
│  4. If the request is INVALID (e.g., minor/senior requesting 10-year passport),    write 'ELIGIBILITY FLAG:'    │
│  followed by a clear explanation and the corrected valid option.                                                │
│  Use the local database as fallback if needed.                                                                  │
│  Agent:                                                                                                         │
│  Bangladesh Passport Policy Expert                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Applicant profile: I am a 35-year-old government employee working at a public university. I need a       │
│  48-page passport with regular delivery. I have my NID card and I live in Dhaka.                                │
│                                                                                                                 │
│  Using ALL outputs from the Policy Guardian, Fee Calculator, and Document Architect, compile the final          │
│  Passport Readiness Report in BOTH English and Bangla.                                                          │
│                                                                                                                 │
│  --- SECTION 1: English Markdown Table ---                                                                      │
│  Two columns: Field | Details.                                                                                  │
│  Rows: Applicant Profile, Eligibility Status, Passport Validity, Required ID Document, Page Count, Delivery     │
│  Type, Delivery Time, Total Fee (BDT VAT Included), Required Documents.                                         │
│  If an ELIGIBILITY FLAG was raised, add it as the FIRST row.                                                    │
│                                                                                                                 │
│  --- SECTION 2: বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট ---                                                                         │
│  Translate the entire table into Bangla. Two columns: ক্ষেত্র | বিবরণ.                                              │
│  All row labels AND all content values must be written in Bangla.                                               │
│  If an eligibility flag exists, first row label must be: যোগ্যতার সমস্যা                                             │
│                                                                                                                 │
│  Output ONLY the two table sections with headings. No extra text outside the tables.                            │
│  ID: 8f5bd563-9a0c-44d7-b6e8-dcd253a54078                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bilingual Passport Report Officer                                                                       │
│                                                                                                                 │
│  Task: Applicant profile: I am a 35-year-old government employee working at a public university. I need a       │
│  48-page passport with regular delivery. I have my NID card and I live in Dhaka.                                │
│                                                                                                                 │
│  Using ALL outputs from the Policy Guardian, Fee Calculator, and Document Architect, compile the final          │
│  Passport Readiness Report in BOTH English and Bangla.                                                          │
│                                                                                                                 │
│  --- SECTION 1: English Markdown Table ---                                                                      │
│  Two columns: Field | Details.                                                                                  │
│  Rows: Applicant Profile, Eligibility Status, Passport Validity, Required ID Document, Page Count, Delivery     │
│  Type, Delivery Time, Total Fee (BDT VAT Included), Required Documents.                                         │
│  If an ELIGIBILITY FLAG was raised, add it as the FIRST row.                                                    │
│                                                                                                                 │
│  --- SECTION 2: বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট ---                                                                         │
│  Translate the entire table into Bangla. Two columns: ক্ষেত্র | বিবরণ.                                              │
│  All row labels AND all content values must be written in Bangla.                                               │
│  If an eligibility flag exists, first row label must be: যোগ্যতার সমস্যা                                             │
│                                                                                                                 │
│  Output ONLY the two table sections with headings. No extra text outside the tables.                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bilingual Passport Report Officer                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### SECTION 1 — English Markdown Table                                                                         │
│  | Field | Details |                                                                                            │
│  | --- | --- |                                                                                                  │
│  | Eligibility Flag |  |                                                                                        │
│  | Applicant Profile | 35-year-old government employee |                                                        │
│  | Eligibility Status | Approved |                                                                              │
│  | Passport Validity | 5 years or 10 years |                                                                    │
│  | Required ID Document | NID Card |                                                                            │
│  | Page Count | 48 pages |                                                                                      │
│  | Delivery Type | Regular |                                                                                    │
│  | Delivery Time | 15 working days |                                                                            │
│  | Total Fee (BDT VAT Included) | 4025 BDT |                                                                    │
│  | Required Documents | NID, No Objection Certificate (NOC) or Government Order (GO), Application Summary,      │
│  Payment Slip |                                                                                                 │
│                                                                                                                 │
│  ### SECTION 2 — বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট                                                                            │
│  | ক্ষেত্র | বিবরণ |                                                                                                │
│  | --- | --- |                                                                                                  │
│  | যোগ্যতার সমস্যা |  |                                                                                              │
│  | আবেদনকারীর প্রোফাইল | ৩৫ বছর বয়সী সরকারি কর্মচারী |                                                                      │
│  | যোগ্যতার অবস্থা | অনুমোদিত |                                                                                         │
│  | পাসপোর্টের মেয়াদ | ৫ বছর বা ১০ বছর |                                                                                │
│  | প্রয়োজনীয় পরিচয়পত্র | এনআইডি কার্ড |                                                                                 │
│  | পৃষ্ঠা সংখ্যা | ৪৮ পৃষ্ঠা |                                                                                           │
│  | ডেলিভারির ধরন | নিয়মিত |                                                                                           │
│  | ডেলিভারির সময় | ১৫ কর্মদিবস |                                                                                      │
│  | মোট ফি (ভ্যাটসহ BDT) | ৪০২৫ BDT |                                                                                │
│  | প্রয়োজনীয় কাগজপত্র | এনআইডি, নো অবজেকশন সার্টিফিকেট (এনওসি) বা সরকারি আদেশ (জিও), আবেদন সারাংশ, পেমেন্ট স্লিপ |                          │
│                                                     

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Applicant profile: I am a 35-year-old government employee working at a public university. I need a 48-page     │
│  passport with regular delivery. I have my NID card and I live in Dhaka.                                        │
│                                                                                                                 │
│  Using ALL outputs from the Policy Guardian, Fee Calculator, and Document Architect, compile the final          │
│  Passport Readiness Report in BOTH English and Bangla.                                                          │
│                                                                                                                 │
│  --- SECTION 1: English Markdown Table ---                                                                      │
│  Two columns: Field | Details.                                                                                  │
│  Rows: Applicant Profile, Eligibility Status, Passport Validity, Required ID Document, Page Count, Delivery     │
│  Type, Delivery Time, Total Fee (BDT VAT Included), Required Documents.                                         │
│  If an ELIGIBILITY FLAG was raised, add it as the FIRST row.                                                    │
│                                                                                                                 │
│  --- SECTION 2: বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট ---                                                                         │
│  Translate the entire table into Bangla. Two columns: ক্ষেত্র | বিবরণ.                                              │
│  All row labels AND all content values must be written in Bangla.                                               │
│  If an eligibility flag exists, first row label must be: যোগ্যতার সমস্যা                                             │
│                                                                                                                 │
│  Output ONLY the two table sections with headings. No extra text outside the tables.                            │
│  Agent:                                                                                                         │
│  Bilingual Passport Report Officer                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f894040f-314d-42da-a7fa-6b550f28bd81                                                                           │
│  Final Output: ### SECTION 1 — English Markdown Table                                                           │
│  | Field | Details |                                                                                            │
│  | --- | --- |                                                                                                  │
│  | Eligibility Flag |  |                                                                                        │
│  | Applicant Profile | 35-year-old government employee |                                                        │
│  | Eligibility Status | Approved |                                                                              │
│  | Passport Validity | 5 years or 10 years |                                                                    │
│  | Required ID Document | NID Card |                                                                            │
│  | Page Count | 48 pages |                                                                                      │
│  | Delivery Type | Regular |                                                                                    │
│  | Delivery Time | 15 working days |                                                                            │
│  | Total Fee (BDT VAT Included) | 4025 BDT |                                                                    │
│  | Required Documents | NID, No Objection Certificate (NOC) or Government Order (GO), Application Summary,      │
│  Payment Slip |                                                                                                 │
│                                                                                                                 │
│  ### SECTION 2 — বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট                                                                            │
│  | ক্ষেত্র | বিবরণ |                                                                                                │
│  | --- | --- |                                                                                                  │
│  | যোগ্যতার সমস্যা |  |                                                                                              │
│  | আবেদনকারীর প্রোফাইল | ৩৫ বছর বয়সী সরকারি কর্মচারী |                                                                      │
│  | যোগ্যতার অবস্থা | অনুমোদিত |                                                                                         │
│  | পাসপোর্টের মেয়াদ | ৫ বছর বা ১০ বছর |                                                                                │
│  | প্রয়োজনীয় পরিচয়পত্র | এনআইডি কার্ড |                                                                                 │
│  | পৃষ্ঠা সংখ্যা | ৪৮ পৃষ্ঠা |                                                                                           │
│  | ডেলিভারির ধরন | নিয়মিত |                                                                                           │
│  | ডেলিভারির সময় | ১৫ কর্মদিবস |                                                                                      │
│  | মোট ফি (ভ্যাটসহ BDT) | ৪০২৫ BDT |                                                 

In [ ]:
display(Markdown(str(result_3)))


### SECTION 1 — English Markdown Table 
| Field | Details |
| --- | --- |
| Eligibility Flag |  |
| Applicant Profile | 35-year-old government employee |
| Eligibility Status | Approved |
| Passport Validity | 5 years or 10 years |
| Required ID Document | NID Card |
| Page Count | 48 pages |
| Delivery Type | Regular |
| Delivery Time | 15 working days |
| Total Fee (BDT VAT Included) | 4025 BDT |
| Required Documents | NID, No Objection Certificate (NOC) or Government Order (GO), Application Summary, Payment Slip |

### SECTION 2 — বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট 
| ক্ষেত্র | বিবরণ |
| --- | --- |
| যোগ্যতার সমস্যা |  |
| আবেদনকারীর প্রোফাইল | ৩৫ বছর বয়সী সরকারি কর্মচারী |
| যোগ্যতার অবস্থা | অনুমোদিত |
| পাসপোর্টের মেয়াদ | ৫ বছর বা ১০ বছর |
| প্রয়োজনীয় পরিচয়পত্র | এনআইডি কার্ড |
| পৃষ্ঠা সংখ্যা | ৪৮ পৃষ্ঠা |
| ডেলিভারির ধরন | নিয়মিত |
| ডেলিভারির সময় | ১৫ কর্মদিবস |
| মোট ফি (ভ্যাটসহ BDT) | ৪০২৫ BDT |
| প্রয়োজনীয় কাগজপত্র | এনআইডি, নো অবজেকশন সার্টিফিকেট (এনওসি) বা সরকারি আদেশ (জিও), আবেদন সারাংশ, পেমেন্ট স্লিপ |

In [ ]:
#Senior Citizen Requesting 10-Year Passport
scenario_4 = (
    "I am a 70-year-old retired person. "
    "I would like a 48-page passport with 10-year validity using regular delivery. "
    "I have my NID card and I live in Chittagong."
)

crew_4 = build_crew(scenario_4)
result_4 = crew_4.kickoff()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  11edf626-b1f6-4c34-a794-ba43c2d03bcd                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Applicant profile: I am a 70-year-old retired person. I would like a 48-page passport with 10-year       │
│  validity using regular delivery. I have my NID card and I live in Chittagong.                                  │
│                                                                                                                 │
│  Based on the applicant age, determine:                                                                         │
│  1. Permitted passport validity: 5 years or 10 years.                                                           │
│     - Under 18: maximum 5 years only. 10-year passport is NOT permitted.                                        │
│     - 18 to 64: both 5-year and 10-year are permitted.                                                          │
│     - 65 and above: maximum 5 years only. 10-year passport is NOT permitted.                                    │
│  2. Required identification: NID Card (adults) or Birth Registration in English (minors).                       │
│  3. Whether the requested validity is allowed for this age group.                                               │
│  4. If the request is INVALID (e.g., minor/senior requesting 10-year passport),    write 'ELIGIBILITY FLAG:'    │
│  followed by a clear explanation and the corrected valid option.                                                │
│  Use the local database as fallback if needed.                                                                  │
│  ID: 275d6fb4-5620-4920-abfb-5d16265c38c8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bangladesh Passport Policy Expert                                                                       │
│                                                                                                                 │
│  Task: Applicant profile: I am a 70-year-old retired person. I would like a 48-page passport with 10-year       │
│  validity using regular delivery. I have my NID card and I live in Chittagong.                                  │
│                                                                                                                 │
│  Based on the applicant age, determine:                                                                         │
│  1. Permitted passport validity: 5 years or 10 years.                                                           │
│     - Under 18: maximum 5 years only. 10-year passport is NOT permitted.                                        │
│     - 18 to 64: both 5-year and 10-year are permitted.                                                          │
│     - 65 and above: maximum 5 years only. 10-year passport is NOT permitted.                                    │
│  2. Required identification: NID Card (adults) or Birth Registration in English (minors).                       │
│  3. Whether the requested validity is allowed for this age group.                                               │
│  4. If the request is INVALID (e.g., minor/senior requesting 10-year passport),    write 'ELIGIBILITY FLAG:'    │
│  followed by a clear explanation and the corrected valid option.                                                │
│  Use the local database as fallback if needed.                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bangladesh Passport Policy Expert                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Based on the applicant's profile:                                                                              │
│                                                                                                                 │
│  1. **Permitted passport validity**: Since the applicant is 70 years old, which is above 65, they are only      │
│  eligible for a 5-year passport validity. A 10-year passport is not permitted for seniors.                      │
│                                                                                                                 │
│  2. **Required identification**: As the applicant is an adult (above 18 years old), the required                │
│  identification document is an NID Card.                                                                        │
│                                                                                                                 │
│  3. **Requested validity and allowance**: The applicant requested a 10-year passport validity, which is not     │
│  allowed for their age group.                                                                                   │
│                                                                                                                 │
│  4. **Eligibility status**: The request for a 10-year passport by a senior is not in line with the eligibility  │
│  criteria.                                                                                                      │
│                                                                                                                 │
│  **ELIGIBILITY FLAG:** The applicant, being 70 years old, is not eligible for a 10-year passport. According to  │
│  the regulations, individuals aged 65 and above are limited to a 5-year passport validity.                      │
│                                                                                                                 │
│  **Corrected valid option:** The applicant can apply for a 5-year validity passport. The required document is   │
│  their NID card.                                                                                                │
│                                                                                                                 │
│  **Final Eligibility Decision:**                                                                                │
│  - Permitted validity: 5 years                                                                                  │
│  - Required ID document: NID Card                                                                               │
│  - Eligibility status: Flagged                                                                                  │
│  - Flag message: ELIGIBILITY FLAG: Individuals aged 65 and above are not eligible for a 10-year passport. The   │
│  maximum validity permitted is 5 years. Please correct the application to request a 5-year passport validity.   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Applicant profile: I am a 70-year-old retired person. I would like a 48-page passport with 10-year validity    │
│  using regular delivery. I have my NID card and I live in Chittagong.                                           │
│                                                                                                                 │
│  Based on the applicant age, determine:                                                                         │
│  1. Permitted passport validity: 5 years or 10 years.                                                           │
│     - Under 18: maximum 5 years only. 10-year passport is NOT permitted.                                        │
│     - 18 to 64: both 5-year and 10-year are permitted.                                                          │
│     - 65 and above: maximum 5 years only. 10-year passport is NOT permitted.                                    │
│  2. Required identification: NID Card (adults) or Birth Registration in English (minors).                       │
│  3. Whether the requested validity is allowed for this age group.                                               │
│  4. If the request is INVALID (e.g., minor/senior requesting 10-year passport),    write 'ELIGIBILITY FLAG:'    │
│  followed by a clear explanation and the corrected valid option.                                                │
│  Use the local database as fallback if needed.                                                                  │
│  Agent:                                                                                                         │
│  Bangladesh Passport Policy Expert                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Applicant profile: I am a 70-year-old retired person. I would like a 48-page passport with 10-year       │
│  validity using regular delivery. I have my NID card and I live in Chittagong.                                  │
│                                                                                                                 │
│  Using ALL outputs from the Policy Guardian, Fee Calculator, and Document Architect, compile the final          │
│  Passport Readiness Report in BOTH English and Bangla.                                                          │
│                                                                                                                 │
│  --- SECTION 1: English Markdown Table ---                                                                      │
│  Two columns: Field | Details.                                                                                  │
│  Rows: Applicant Profile, Eligibility Status, Passport Validity, Required ID Document, Page Count, Delivery     │
│  Type, Delivery Time, Total Fee (BDT VAT Included), Required Documents.                                         │
│  If an ELIGIBILITY FLAG was raised, add it as the FIRST row.                                                    │
│                                                                                                                 │
│  --- SECTION 2: বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট ---                                                                         │
│  Translate the entire table into Bangla. Two columns: ক্ষেত্র | বিবরণ.                                              │
│  All row labels AND all content values must be written in Bangla.                                               │
│  If an eligibility flag exists, first row label must be: যোগ্যতার সমস্যা                                             │
│                                                                                                                 │
│  Output ONLY the two table sections with headings. No extra text outside the tables.                            │
│  ID: a8b3df73-6cab-4dc5-a6ca-f9d60ed4f135                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bilingual Passport Report Officer                                                                       │
│                                                                                                                 │
│  Task: Applicant profile: I am a 70-year-old retired person. I would like a 48-page passport with 10-year       │
│  validity using regular delivery. I have my NID card and I live in Chittagong.                                  │
│                                                                                                                 │
│  Using ALL outputs from the Policy Guardian, Fee Calculator, and Document Architect, compile the final          │
│  Passport Readiness Report in BOTH English and Bangla.                                                          │
│                                                                                                                 │
│  --- SECTION 1: English Markdown Table ---                                                                      │
│  Two columns: Field | Details.                                                                                  │
│  Rows: Applicant Profile, Eligibility Status, Passport Validity, Required ID Document, Page Count, Delivery     │
│  Type, Delivery Time, Total Fee (BDT VAT Included), Required Documents.                                         │
│  If an ELIGIBILITY FLAG was raised, add it as the FIRST row.                                                    │
│                                                                                                                 │
│  --- SECTION 2: বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট ---                                                                         │
│  Translate the entire table into Bangla. Two columns: ক্ষেত্র | বিবরণ.                                              │
│  All row labels AND all content values must be written in Bangla.                                               │
│  If an eligibility flag exists, first row label must be: যোগ্যতার সমস্যা                                             │
│                                                                                                                 │
│  Output ONLY the two table sections with headings. No extra text outside the tables.                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bilingual Passport Report Officer                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### SECTION 1 — English Markdown Table                                                                         │
│  | Field | Details |                                                                                            │
│  | --- | --- |                                                                                                  │
│  | Eligibility Flag | ELIGIBILITY FLAG: Individuals aged 65 and above are not eligible for a 10-year passport.  │
│  The maximum validity permitted is 5 years. |                                                                   │
│  | Applicant Profile | 70-year-old retired person |                                                             │
│  | Eligibility Status | Flagged |                                                                               │
│  | Passport Validity | 5 years (Corrected from 10 years) |                                                      │
│  | Required ID Document | NID Card |                                                                            │
│  | Page Count | 48 pages |                                                                                      │
│  | Delivery Type | Regular |                                                                                    │
│  | Delivery Time | 15 working days |                                                                            │
│  | Total Fee (BDT VAT Included) | 4025 BDT |                                                                    │
│  | Required Documents | NID, Application Summary, Payment Slip |                                                │
│                                                                                                                 │
│  ### SECTION 2 — বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট                                                                            │
│  | ক্ষেত্র | বিবরণ |                                                                                                │
│  | --- | --- |                                                                                                  │
│  | যোগ্যতার সমস্যা | যোগ্যতার সমস্যা: ৬৫ বছরের বেশি বয়সী ব্যক্তিরা ১০ বছরের পাসপোর্টের জন্য যোগ্য নয়। সর্বোচ্চ অনুমোদিত মেয়াদ ৫ বছর। |            │
│  | আবেদনকারীর প্রোফাইল | ৭০ বছর বয়সী অবসরপ্রাপ্ত ব্যক্তি |                                                                   │
│  | যোগ্যতার অবস্থা | ফ্ল্যাগযুক্ত |                                                                                       │
│  | পাসপোর্টের মেয়াদ | ৫ বছর (১০ বছর থেকে সংশোধিত) |                                                                        │
│  | প্রয়োজনীয় পরিচয়পত্র | এনআইডি কার্ড |                                                                                 │
│  | পৃষ্ঠা সংখ্যা | ৪৮ পৃষ্ঠা |                                                                                           │
│  | ডেলিভারির ধরন | নিয়মিত |                                                                                           │
│  | ডেলিভারির সময় | ১৫ কর্মদিবস |                                                                                      │
│  | মোট ফি (ভ্যাটসহ BDT) | ৪০২৫ BDT |                                                                                │
│  | প্রয়োজনীয় কাগজপত্র | এনআইডি, আবেদন সারাংশ, পেমেন্ট স্লিপ |                                                                  │
│                           

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Applicant profile: I am a 70-year-old retired person. I would like a 48-page passport with 10-year validity    │
│  using regular delivery. I have my NID card and I live in Chittagong.                                           │
│                                                                                                                 │
│  Using ALL outputs from the Policy Guardian, Fee Calculator, and Document Architect, compile the final          │
│  Passport Readiness Report in BOTH English and Bangla.                                                          │
│                                                                                                                 │
│  --- SECTION 1: English Markdown Table ---                                                                      │
│  Two columns: Field | Details.                                                                                  │
│  Rows: Applicant Profile, Eligibility Status, Passport Validity, Required ID Document, Page Count, Delivery     │
│  Type, Delivery Time, Total Fee (BDT VAT Included), Required Documents.                                         │
│  If an ELIGIBILITY FLAG was raised, add it as the FIRST row.                                                    │
│                                                                                                                 │
│  --- SECTION 2: বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট ---                                                                         │
│  Translate the entire table into Bangla. Two columns: ক্ষেত্র | বিবরণ.                                              │
│  All row labels AND all content values must be written in Bangla.                                               │
│  If an eligibility flag exists, first row label must be: যোগ্যতার সমস্যা                                             │
│                                                                                                                 │
│  Output ONLY the two table sections with headings. No extra text outside the tables.                            │
│  Agent:                                                                                                         │
│  Bilingual Passport Report Officer                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  11edf626-b1f6-4c34-a794-ba43c2d03bcd                                                                           │
│  Final Output: ### SECTION 1 — English Markdown Table                                                           │
│  | Field | Details |                                                                                            │
│  | --- | --- |                                                                                                  │
│  | Eligibility Flag | ELIGIBILITY FLAG: Individuals aged 65 and above are not eligible for a 10-year passport.  │
│  The maximum validity permitted is 5 years. |                                                                   │
│  | Applicant Profile | 70-year-old retired person |                                                             │
│  | Eligibility Status | Flagged |                                                                               │
│  | Passport Validity | 5 years (Corrected from 10 years) |                                                      │
│  | Required ID Document | NID Card |                                                                            │
│  | Page Count | 48 pages |                                                                                      │
│  | Delivery Type | Regular |                                                                                    │
│  | Delivery Time | 15 working days |                                                                            │
│  | Total Fee (BDT VAT Included) | 4025 BDT |                                                                    │
│  | Required Documents | NID, Application Summary, Payment Slip |                                                │
│                                                                                                                 │
│  ### SECTION 2 — বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট                                                                            │
│  | ক্ষেত্র | বিবরণ |                                                                                                │
│  | --- | --- |                                                                                                  │
│  | যোগ্যতার সমস্যা | যোগ্যতার সমস্যা: ৬৫ বছরের বেশি বয়সী ব্যক্তিরা ১০ বছরের পাসপোর্টের জন্য যোগ্য নয়। সর্বোচ্চ অনুমোদিত মেয়াদ ৫ বছর। |            │
│  | আবেদনকারীর প্রোফাইল | ৭০ বছর বয়সী অবসরপ্রাপ্ত ব্যক্তি |                                                                   │
│  | যোগ্যতার অবস্থা | ফ্ল্যাগযুক্ত |                                                                                       │
│  | পাসপোর্টের মেয়াদ | ৫ বছর (১০ বছর থেকে সংশোধিত) |                                                                        │
│  | প্রয়োজনীয় পরিচয়পত্র | এনআইডি কার্ড |                                                                                 │
│  | পৃষ্ঠা সংখ্যা | ৪৮ পৃষ্ঠা |                                                                                           │
│  | ডেলিভারির ধরন | নিয়মিত |                                                                                           │
│  | ডেলিভারির সময় | ১৫ কর্মদিবস |                                                                                      │
│  | মোট ফি (ভ্যাটসহ BDT) | ৪০২৫ BDT |          

In [ ]:
display(Markdown(str(result_4)))


### SECTION 1 — English Markdown Table 
| Field | Details |
| --- | --- |
| Eligibility Flag | ELIGIBILITY FLAG: Individuals aged 65 and above are not eligible for a 10-year passport. The maximum validity permitted is 5 years. |
| Applicant Profile | 70-year-old retired person |
| Eligibility Status | Flagged |
| Passport Validity | 5 years (Corrected from 10 years) |
| Required ID Document | NID Card |
| Page Count | 48 pages |
| Delivery Type | Regular |
| Delivery Time | 15 working days |
| Total Fee (BDT VAT Included) | 4025 BDT |
| Required Documents | NID, Application Summary, Payment Slip |

### SECTION 2 — বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট 
| ক্ষেত্র | বিবরণ |
| --- | --- |
| যোগ্যতার সমস্যা | যোগ্যতার সমস্যা: ৬৫ বছরের বেশি বয়সী ব্যক্তিরা ১০ বছরের পাসপোর্টের জন্য যোগ্য নয়। সর্বোচ্চ অনুমোদিত মেয়াদ ৫ বছর। |
| আবেদনকারীর প্রোফাইল | ৭০ বছর বয়সী অবসরপ্রাপ্ত ব্যক্তি |
| যোগ্যতার অবস্থা | ফ্ল্যাগযুক্ত |
| পাসপোর্টের মেয়াদ | ৫ বছর (১০ বছর থেকে সংশোধিত) |
| প্রয়োজনীয় পরিচয়পত্র | এনআইডি কার্ড |
| পৃষ্ঠা সংখ্যা | ৪৮ পৃষ্ঠা |
| ডেলিভারির ধরন | নিয়মিত |
| ডেলিভারির সময় | ১৫ কর্মদিবস |
| মোট ফি (ভ্যাটসহ BDT) | ৪০২৫ BDT |
| প্রয়োজনীয় কাগজপত্র | এনআইডি, আবেদন সারাংশ, পেমেন্ট স্লিপ |

In [ ]:
#Using my data
custom_input = (
    "I am a 25-year-old private University Student. "
    "I need a 64-page passport with super express delivery because I have an emergency trip. "
    "I have my NID card and I live in Dhaka."
)

crew_custom = build_crew(custom_input)
result_custom = crew_custom.kickoff()


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  cbcfdc9c-5b84-43d1-8e68-478023027f07                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Applicant profile: I am a 25-year-old private University Student. I need a 64-page passport with super   │
│  express delivery because I have an emergency trip. I have my NID card and I live in Dhaka.                     │
│                                                                                                                 │
│  Based on the applicant age, determine:                                                                         │
│  1. Permitted passport validity: 5 years or 10 years.                                                           │
│     - Under 18: maximum 5 years only. 10-year passport is NOT permitted.                                        │
│     - 18 to 64: both 5-year and 10-year are permitted.                                                          │
│     - 65 and above: maximum 5 years only. 10-year passport is NOT permitted.                                    │
│  2. Required identification: NID Card (adults) or Birth Registration in English (minors).                       │
│  3. Whether the requested validity is allowed for this age group.                                               │
│  4. If the request is INVALID (e.g., minor/senior requesting 10-year passport),    write 'ELIGIBILITY FLAG:'    │
│  followed by a clear explanation and the corrected valid option.                                                │
│  Use the local database as fallback if needed.                                                                  │
│  ID: 05b60917-a0cb-4d02-bb53-887ef47e5558                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bangladesh Passport Policy Expert                                                                       │
│                                                                                                                 │
│  Task: Applicant profile: I am a 25-year-old private University Student. I need a 64-page passport with super   │
│  express delivery because I have an emergency trip. I have my NID card and I live in Dhaka.                     │
│                                                                                                                 │
│  Based on the applicant age, determine:                                                                         │
│  1. Permitted passport validity: 5 years or 10 years.                                                           │
│     - Under 18: maximum 5 years only. 10-year passport is NOT permitted.                                        │
│     - 18 to 64: both 5-year and 10-year are permitted.                                                          │
│     - 65 and above: maximum 5 years only. 10-year passport is NOT permitted.                                    │
│  2. Required identification: NID Card (adults) or Birth Registration in English (minors).                       │
│  3. Whether the requested validity is allowed for this age group.                                               │
│  4. If the request is INVALID (e.g., minor/senior requesting 10-year passport),    write 'ELIGIBILITY FLAG:'    │
│  followed by a clear explanation and the corrected valid option.                                                │
│  Use the local database as fallback if needed.                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bangladesh Passport Policy Expert                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Based on the applicant's profile:                                                                              │
│                                                                                                                 │
│  1. **Permitted passport validity**: Since the applicant is 25 years old, which falls within the 18 to 64 age   │
│  range, they are eligible for either a 5-year or 10-year passport validity.                                     │
│                                                                                                                 │
│  2. **Required identification**: As the applicant is an adult (above 18 years old), the required                │
│  identification document is an NID Card.                                                                        │
│                                                                                                                 │
│  3. **Requested validity and allowance**: The applicant did not specify the requested validity but is eligible  │
│  for both. The request for a 64-page passport with super express delivery due to an emergency trip seems        │
│  standard and does not inherently conflict with eligibility criteria.                                           │
│                                                                                                                 │
│  4. **Eligibility status**: The applicant's request seems to be generally in line with the eligibility          │
│  criteria based on age.                                                                                         │
│                                                                                                                 │
│  Given the provided information and standard criteria:                                                          │
│                                                                                                                 │
│  - **Permitted validity**: 5 years or 10 years                                                                  │
│  - **Required ID document**: NID Card                                                                           │
│  - **Eligibility status**: Approved                                                                             │
│  - **Flag message**: None                                                                                       │
│                                                                                                                 │
│  **Final Eligibility Decision:**                                                                                │
│  - Permitted validity: 5 years or 10 years                                                                      │
│  - Required ID document: NID Card                                                                               │
│  - Eligibility status: Approved                                                                                 │
│  - Flag message: None                                                                                           │
│                                                                                                                 │
│  The applicant, being 25 years old and having an NID, is eligible for a passport with either 5-year or 10-year  │
│  validity. Their request for super express delivery due

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Applicant profile: I am a 25-year-old private University Student. I need a 64-page passport with super         │
│  express delivery because I have an emergency trip. I have my NID card and I live in Dhaka.                     │
│                                                                                                                 │
│  Based on the applicant age, determine:                                                                         │
│  1. Permitted passport validity: 5 years or 10 years.                                                           │
│     - Under 18: maximum 5 years only. 10-year passport is NOT permitted.                                        │
│     - 18 to 64: both 5-year and 10-year are permitted.                                                          │
│     - 65 and above: maximum 5 years only. 10-year passport is NOT permitted.                                    │
│  2. Required identification: NID Card (adults) or Birth Registration in English (minors).                       │
│  3. Whether the requested validity is allowed for this age group.                                               │
│  4. If the request is INVALID (e.g., minor/senior requesting 10-year passport),    write 'ELIGIBILITY FLAG:'    │
│  followed by a clear explanation and the corrected valid option.                                                │
│  Use the local database as fallback if needed.                                                                  │
│  Agent:                                                                                                         │
│  Bangladesh Passport Policy Expert                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Applicant profile: I am a 25-year-old private University Student. I need a 64-page passport with super   │
│  express delivery because I have an emergency trip. I have my NID card and I live in Dhaka.                     │
│                                                                                                                 │
│  Using ALL outputs from the Policy Guardian, Fee Calculator, and Document Architect, compile the final          │
│  Passport Readiness Report in BOTH English and Bangla.                                                          │
│                                                                                                                 │
│  --- SECTION 1: English Markdown Table ---                                                                      │
│  Two columns: Field | Details.                                                                                  │
│  Rows: Applicant Profile, Eligibility Status, Passport Validity, Required ID Document, Page Count, Delivery     │
│  Type, Delivery Time, Total Fee (BDT VAT Included), Required Documents.                                         │
│  If an ELIGIBILITY FLAG was raised, add it as the FIRST row.                                                    │
│                                                                                                                 │
│  --- SECTION 2: বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট ---                                                                         │
│  Translate the entire table into Bangla. Two columns: ক্ষেত্র | বিবরণ.                                              │
│  All row labels AND all content values must be written in Bangla.                                               │
│  If an eligibility flag exists, first row label must be: যোগ্যতার সমস্যা                                             │
│                                                                                                                 │
│  Output ONLY the two table sections with headings. No extra text outside the tables.                            │
│  ID: a54d1116-7a4e-4385-b691-d21ab563c106                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bilingual Passport Report Officer                                                                       │
│                                                                                                                 │
│  Task: Applicant profile: I am a 25-year-old private University Student. I need a 64-page passport with super   │
│  express delivery because I have an emergency trip. I have my NID card and I live in Dhaka.                     │
│                                                                                                                 │
│  Using ALL outputs from the Policy Guardian, Fee Calculator, and Document Architect, compile the final          │
│  Passport Readiness Report in BOTH English and Bangla.                                                          │
│                                                                                                                 │
│  --- SECTION 1: English Markdown Table ---                                                                      │
│  Two columns: Field | Details.                                                                                  │
│  Rows: Applicant Profile, Eligibility Status, Passport Validity, Required ID Document, Page Count, Delivery     │
│  Type, Delivery Time, Total Fee (BDT VAT Included), Required Documents.                                         │
│  If an ELIGIBILITY FLAG was raised, add it as the FIRST row.                                                    │
│                                                                                                                 │
│  --- SECTION 2: বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট ---                                                                         │
│  Translate the entire table into Bangla. Two columns: ক্ষেত্র | বিবরণ.                                              │
│  All row labels AND all content values must be written in Bangla.                                               │
│  If an eligibility flag exists, first row label must be: যোগ্যতার সমস্যা                                             │
│                                                                                                                 │
│  Output ONLY the two table sections with headings. No extra text outside the tables.                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bilingual Passport Report Officer                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### SECTION 1 — English Markdown Table                                                                         │
│  | Field | Details |                                                                                            │
│  | --- | --- |                                                                                                  │
│  | Eligibility Flag |  |                                                                                        │
│  | Applicant Profile | 25-year-old private University Student |                                                 │
│  | Eligibility Status | Approved |                                                                              │
│  | Passport Validity | 5 years or 10 years |                                                                    │
│  | Required ID Document | NID Card |                                                                            │
│  | Page Count | 64 pages |                                                                                      │
│  | Delivery Type | Super Express |                                                                              │
│  | Delivery Time | 2 working days |                                                                             │
│  | Total Fee (BDT VAT Included) | 12075 BDT |                                                                   │
│  | Required Documents | NID, Application Summary, Payment Slip, Letter from University or Affidavit stating     │
│  student status (optional) |                                                                                    │
│                                                                                                                 │
│  ### SECTION 2 — বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট                                                                            │
│  | ক্ষেত্র | বিবরণ |                                                                                                │
│  | --- | --- |                                                                                                  │
│  | যোগ্যতার সমস্যা |  |                                                                                              │
│  | আবেদনকারীর প্রোফাইল | ২৫ বছর বয়সী বেসরকারী বিশ্ববিদ্যালয়ের ছাত্র |                                                             │
│  | যোগ্যতার অবস্থা | অনুমোদিত |                                                                                         │
│  | পাসপোর্টের মেয়াদ | ৫ বছর বা ১০ বছর |                                                                                │
│  | প্রয়োজনীয় পরিচয়পত্র | এনআইডি কার্ড |                                                                                 │
│  | পৃষ্ঠা সংখ্যা | ৬৪ পৃষ্ঠা |                                                                                           │
│  | ডেলিভারির ধরন | সুপার এক্সপ্রেস |                                                                                     │
│  | ডেলিভারির সময় | ২ কর্মদিবস |                                                                                       │
│  | মোট ফি (ভ্যাটসহ BDT) | ১২০৭৫ BDT |                                                                               │
│  | প্রয়োজনীয় কাগজপত্র | এনআইডি, আবেদন সারাংশ, পেমেন্ট স্লিপ, বিশ্ববিদ্যালয়ের চিঠি বা ছাত্র অবস্থা সম্পর্কিত হলিউড (ঐচ্ছিক) |                      │
│                                     

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Applicant profile: I am a 25-year-old private University Student. I need a 64-page passport with super         │
│  express delivery because I have an emergency trip. I have my NID card and I live in Dhaka.                     │
│                                                                                                                 │
│  Using ALL outputs from the Policy Guardian, Fee Calculator, and Document Architect, compile the final          │
│  Passport Readiness Report in BOTH English and Bangla.                                                          │
│                                                                                                                 │
│  --- SECTION 1: English Markdown Table ---                                                                      │
│  Two columns: Field | Details.                                                                                  │
│  Rows: Applicant Profile, Eligibility Status, Passport Validity, Required ID Document, Page Count, Delivery     │
│  Type, Delivery Time, Total Fee (BDT VAT Included), Required Documents.                                         │
│  If an ELIGIBILITY FLAG was raised, add it as the FIRST row.                                                    │
│                                                                                                                 │
│  --- SECTION 2: বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট ---                                                                         │
│  Translate the entire table into Bangla. Two columns: ক্ষেত্র | বিবরণ.                                              │
│  All row labels AND all content values must be written in Bangla.                                               │
│  If an eligibility flag exists, first row label must be: যোগ্যতার সমস্যা                                             │
│                                                                                                                 │
│  Output ONLY the two table sections with headings. No extra text outside the tables.                            │
│  Agent:                                                                                                         │
│  Bilingual Passport Report Officer                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  cbcfdc9c-5b84-43d1-8e68-478023027f07                                                                           │
│  Final Output: ### SECTION 1 — English Markdown Table                                                           │
│  | Field | Details |                                                                                            │
│  | --- | --- |                                                                                                  │
│  | Eligibility Flag |  |                                                                                        │
│  | Applicant Profile | 25-year-old private University Student |                                                 │
│  | Eligibility Status | Approved |                                                                              │
│  | Passport Validity | 5 years or 10 years |                                                                    │
│  | Required ID Document | NID Card |                                                                            │
│  | Page Count | 64 pages |                                                                                      │
│  | Delivery Type | Super Express |                                                                              │
│  | Delivery Time | 2 working days |                                                                             │
│  | Total Fee (BDT VAT Included) | 12075 BDT |                                                                   │
│  | Required Documents | NID, Application Summary, Payment Slip, Letter from University or Affidavit stating     │
│  student status (optional) |                                                                                    │
│                                                                                                                 │
│  ### SECTION 2 — বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট                                                                            │
│  | ক্ষেত্র | বিবরণ |                                                                                                │
│  | --- | --- |                                                                                                  │
│  | যোগ্যতার সমস্যা |  |                                                                                              │
│  | আবেদনকারীর প্রোফাইল | ২৫ বছর বয়সী বেসরকারী বিশ্ববিদ্যালয়ের ছাত্র |                                                             │
│  | যোগ্যতার অবস্থা | অনুমোদিত |                                                                                         │
│  | পাসপোর্টের মেয়াদ | ৫ বছর বা ১০ বছর |                                                                                │
│  | প্রয়োজনীয় পরিচয়পত্র | এনআইডি কার্ড |                                                                                 │
│  | পৃষ্ঠা সংখ্যা | ৬৪ পৃষ্ঠা |                                                                                           │
│  | ডেলিভারির ধরন | সুপার এক্সপ্রেস |                                                                                     │
│  | ডেলিভারির সময় | ২ কর্মদিবস |                                                                                       │
│  | মোট ফি (ভ্যাটসহ BDT) | ১২০৭৫ BDT |                                       

In [ ]:
display(Markdown(str(result_custom)))

### SECTION 1 — English Markdown Table 
| Field | Details |
| --- | --- |
| Eligibility Flag |  |
| Applicant Profile | 25-year-old private University Student |
| Eligibility Status | Approved |
| Passport Validity | 5 years or 10 years |
| Required ID Document | NID Card |
| Page Count | 64 pages |
| Delivery Type | Super Express |
| Delivery Time | 2 working days |
| Total Fee (BDT VAT Included) | 12075 BDT |
| Required Documents | NID, Application Summary, Payment Slip, Letter from University or Affidavit stating student status (optional) |

### SECTION 2 — বাংলা পাসপোর্ট প্রস্তুতি রিপোর্ট 
| ক্ষেত্র | বিবরণ |
| --- | --- |
| যোগ্যতার সমস্যা |  |
| আবেদনকারীর প্রোফাইল | ২৫ বছর বয়সী বেসরকারী বিশ্ববিদ্যালয়ের ছাত্র |
| যোগ্যতার অবস্থা | অনুমোদিত |
| পাসপোর্টের মেয়াদ | ৫ বছর বা ১০ বছর |
| প্রয়োজনীয় পরিচয়পত্র | এনআইডি কার্ড |
| পৃষ্ঠা সংখ্যা | ৬৪ পৃষ্ঠা |
| ডেলিভারির ধরন | সুপার এক্সপ্রেস |
| ডেলিভারির সময় | ২ কর্মদিবস |
| মোট ফি (ভ্যাটসহ BDT) | ১২০৭৫ BDT |
| প্রয়োজনীয় কাগজপত্র | এনআইডি, আবেদন সারাংশ, পেমেন্ট স্লিপ, বিশ্ববিদ্যালয়ের চিঠি বা ছাত্র অবস্থা সম্পর্কিত হলিউড (ঐচ্ছিক) |